# Title: Segmetation Data set 
it is a complex of codes to extraxt data from the collected data of streetscape of Isfahan , a famous historic city in Iran, the information are used to recognised better the indexes of human perception. 

In [6]:
import sys
sys.executable


'/Users/hessam/Documents/GitHub/StreetPerception_ML/.venv/bin/python'

In [7]:
import numpy as np
import matplotlib.pyplot as plt

In [8]:
import sys
print(sys.executable)

import numpy
import pandas
import sklearn


/Users/hessam/Documents/GitHub/StreetPerception_ML/.venv/bin/python


In [9]:
#  we use Standard library as below 
import os
from pathlib import Path

# for  Data handling
import numpy as np
import pandas as pd

# for visualization 
import matplotlib.pyplot as plt
import seaborn as sns

# for machine learning 
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler


In [10]:
import pandas as pd

class DataAnalyzer:
    def __init__(self, filepath):
        self.df = pd.read_csv(filepath)

    def add_multiple_merges(self, merge_dict):
        for new_col, cols_to_merge in merge_dict.items():
            self.df[new_col] = self.df[cols_to_merge].sum(axis=1)
    def drop_columns(self, columns_to_drop):
        """
        Drop one or more columns safely from the DataFrame.
        Example: analyzer.drop_columns(["Window", "Door"])
        """
        self.df.drop(columns=columns_to_drop, errors="ignore", inplace=True)
        print(f"Dropped columns: {columns_to_drop}")
        return self.df

    def save_to_csv(self, output_path):
        self.df.to_csv(output_path, index=False)
        print(f"Saved CSV to {output_path}")

    def save_to_excel(self, output_path):
        self.df.to_excel(output_path, index=False)
        print(f" Saved Excel to {output_path}")


# Use the correct class here
analyzer = DataAnalyzer("/kaggle/input/segmentation-excel/segmentation pixel .csv")

# Example merges
merge_dict = {
    "merge_E_H_N": [" Bench and Food stands", "Bulletin board", "Symbol and Sign"],
    "merge_B_C_D_": ["Window", "Door"],
    "Wall_pixels" : [
    "Brick wall with fence",
    "Wall",
    "Steps",
    "Buildings wall",
    "Buildings with column",
    "Metal Fence"
]
}

columns_to_drop= (
    "Brick wall with fence",
    "Wall",
    "Steps",
    "Buildings wall",
    "Buildings with column",
    "Metal Fence")

# Add ALL merges from dictionary
analyzer.add_multiple_merges(merge_dict)
analyzer.drop_columns(columns_to_drop)

# Save outputs
analyzer.save_to_excel("/kaggle/working/merged_output.xlsx")
analyzer.save_to_csv("/kaggle/working/merged_output.csv")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/segmentation-excel/segmentation pixel .csv'

In [ ]:
analyzer.df.head()

In [ ]:
# List of column groups
wall_cols = [
    "Brick wall with fence", "Wall", "Steps",
    "Buildings wall", "Buildings with column", "Metal Fence"
]

ehn_cols = [" Bench and Food stands", "Bulletin board", "Symbol and Sign"]
bd_cols = ["Window", "Door"]
extra_cols = ["Class_34", "group_id"]

cols_to_drop = wall_cols + ehn_cols + bd_cols + extra_cols

# Safely drop columns ONLY if they exist
df_clean = analyzer.df.drop(columns=cols_to_drop, errors="ignore")

# If analyzer.drop_columns() expects a DataFrame, update it
analyzer.df = df_clean


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)


In [ ]:
analyzer.df.head()

In [ ]:
analyzer.save_to_excel("/kaggle/working/droped_output.xlsx")
analyzer.save_to_csv("/kaggle/working/droped_output.csv")

In [ ]:
BASE_DIR = Path("/kaggle/working")
DATA_PATH = BASE_DIR / "droped_output.csv"

df = pd.read_csv(DATA_PATH)

# Categorizing Data Frame based on Street Names from CSV file from sample 1 (first picture) to 193 (the last picture)


In [ ]:
## samples 1 to 15 from AmadMid Street
analyzer = DataAnalyzer("/kaggle/working/droped_output.csv")
df = analyzer.df

# Drop the first row (index 0) and reset index
df = df.drop(0).reset_index(drop=True)

# Drop the first column
df = df.iloc[:, 1:]


# Select only rows 2 to 15 (Python index is zero-based, so rows 1 to 14)
amadmid_df = df.iloc[1:15, :]

# Show this subset
display(amadmid_df)

# Now calculate column sums for this subset (numeric columns only)
column_sums = amadmid_df.select_dtypes(include='number').sum()

# Convert to a clean DataFrame
column_sums_df = pd.DataFrame(column_sums).T
display(column_sums_df)

# Making normalized DataFrame (df_ratio_)


In [ ]:
## Through Compute the mean ratio of each feature pixel across all other pixel features of one photo.
### Each row = one photo
### Each column = one class (greenery, wall, sky, etc.)
### Each cell = ratio of that class in that photo
### Then go down across column( which defined as df.iloc[1:15, :])
#### samples 1 to 15 from AmadMid Street

df_ratio_15 = amadmid_df.div(amadmid_df.sum(axis=1), axis=0)

# df=This is your original DataFrame 
# axis=1: each row = one photo,axis=1 means "move horizontally across columns
# axis=0: each column = one pixel class (e.g., greenery, building, sky, etc.).
# axis=0: mean "move vertically down rows
# df_ratio: each value represents a ratio or percentage of that class (e.g., trees or buildings)
# in the total pixels of the same photo 

In [ ]:
## samples 1 to 15 from AmadMid Street
display(df_ratio_15)

# Propagating means (Mean Ratio) to be used in Correlation theory 
###  by dividing rows of ratio into smaller group and then get .mean from them to be used in Pearson correlation.Pearson Correlation works in large and at least 20 to 30  data set. 

In [ ]:
## samples 1 to 15 from AmadMid Street
#we need to group within DataFrame by artificial “group_id” to create smaller chunks.
#It makes more Numeric.mean  among ratio Data Frame to be used in Pearson Correlation.  


groups = np.arange(len(df_ratio_15)) // 5

street_part_means_15 = (
    df_ratio_15.groupby(groups)
    .mean(numeric_only=True)
    .reset_index(drop=True)
)


street_part_means_15


In [ ]:
## samples 1 to 15 from AmadMid Street
# it is to melt (reshape) the wide dataframe into long format
street_part_means_long_15 = street_part_means_15.melt(
    var_name="Feature",          # name for feature column
    value_name="Mean_Ratio"      # name for ratio column
)

display(street_part_means_long_15)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

wedges, texts, autotexts = ax.pie(
    street_mean_ratios_15,
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.75
)

ax.legend(
    wedges,
    street_mean_ratios_15.index,
    title="Classes",
    loc="center left",
    bbox_to_anchor=(1, 0.5)
)

ax.set_title("Mean Visual Composition of Street")
plt.tight_layout()
plt.show()



In [ ]:
## samples 1 to 15 from AmadMid Street
street_part_means_long_15.to_csv("street_part_means_long_15.csv", index=False)


In [ ]:
street_mean_ratios_15 = df_ratio_15.mean(axis=0)
street_mean_ratios_15

In [ ]:
analyzer = DataAnalyzer("/kaggle/working/droped_output.csv")
df = analyzer.df

# Drop the first row (index 0) and reset index
df = df.drop(0).reset_index(drop=True)

# Drop the first column
df = df.iloc[:, 1:]

# Select only rows 2 to 15 (Python index is zero-based, so rows 1 to 14)
AmadR_df = df.iloc[16:34, :]

# Show this subset
display(AmadR_df)

# Now calculate column sums for this subset (numeric columns only)
column_sums = AmadR_df.select_dtypes(include='number').sum()

# Convert to a clean DataFrame
column_sums_df = pd.DataFrame(column_sums).T
display(column_sums_df)

In [ ]:
df_ratio_34 = AmadR_df.div(AmadR_df.sum(axis=1), axis=0)


In [ ]:
display(df_ratio_34)

# Propagating means (Mean Ratio) to be used in Correlation theory 


In [ ]:
#we need to group within DataFrame by artificial “group_id” to create smaller chunks.
#It makes more Numeric.mean  among ratio Data Frame to be used in Pearson Correlation.  


# Suppose df_ratio_34 is one street with 15 photos (rows)
#This line divides your rows into groups of 5
# Compute mean ratios for each group of 5 photos

groups = np.arange(len(df_ratio_34)) // 5

street_part_means_34 = (
    df_ratio_34.groupby(groups)
    .mean(numeric_only=True)
    .reset_index(drop=True)
)



print(street_part_means_34)


In [ ]:
street_mean_ratio_34 = df_ratio_34.mean(axis=0)


In [ ]:
display(street_mean_ratio_34)


In [ ]:

# Plot a pie chart
street_mean_ratio_34.plot(
    kind='pie',
    autopct='%1.1f%%',       # Show percentages
    figsize=(6, 6),
    startangle=90,           # Rotate so first slice starts at top
    ylabel=''                # Remove y-axis label
)

plt.title("Mean Visual Composition of Street")
plt.show()


In [ ]:
analyzer = DataAnalyzer("/kaggle/working/droped_output.csv")
df = analyzer.df
# Drop the first row (index 0) and reset index
df = df.drop(0).reset_index(drop=True)

# Drop the first column
df = df.iloc[:, 1:]
#The code df = df.iloc[:, 1:] in Pandas selects all rows and all columns except the first one from a 
#DataFrame, then reassigns the result to the df variable, effectively dropping the first column. 
#The : selects all rows, and 1: selects all columns starting from the second one (index 1), based on
#their integer position. 

# Select only rows 34 to 43 (Python index is zero-based, so rows 34 to 42)
ChbaghDarDol_df = df.iloc[34:43, :]

# Show this subset
display(ChbaghDarDol_df)

# Now calculate column sums for this subset (numeric columns only)
column_sums = ChbaghDarDol_df.select_dtypes(include='number').sum()

# Convert to a clean DataFrame
column_sums_df = pd.DataFrame(column_sums).T
display(column_sums_df)

In [ ]:
df_ratio43 = ChbaghDarDol_df.div(ChbaghDarDol_df.sum(axis=1), axis=0)

#df=This is your original DataFrame 
# axis=1: each row = one photo,axis=1 means "move horizontally across columns
# axis=0: each column = one pixel class (e.g., greenery, building, sky, etc.).
# axis=0: mean "move vertically down rows
# df_ratio: each value represents a ratio or percentage of that class (e.g., trees or buildings)
#in the total pixels of the same photo


In [ ]:
df_ratio43.head()

In [ ]:
street_mean_ratio43 = df_ratio43.mean(axis=0)


In [ ]:
display(street_mean_ratio43)


In [ ]:

# Plot a pie chart
street_mean_ratio43.plot(
    kind='pie',
    autopct='% 45.1f%%',       # Show percentages
    figsize=(7, 7),
    startangle=60,           # Rotate so first slice starts at top
    ylabel=''                # Remove y-axis label
)

plt.title("Mean Visual Composition of Street")
plt.show()


In [ ]:


# Suppose df_ratio_15 is one street with 15 photos (rows)
df_ratio43["group_id"] = np.arange(len(df_ratio43)) // 5  # groups of 7 rows
#This line divides your rows into groups of 5
# Compute mean ratios for each group of 5 photos
street_part_means43 = (
    df_ratio43.groupby("group_id")
    .mean(numeric_only=True)
    .reset_index()
)

street_part_means43

In [ ]:
# # Drop the first row (index 0) and reset index
df = df.drop(0).reset_index(drop=True)

# Drop the first column
df = df.iloc[:, 1:]

# Select only rows 43 to 66 (Python index is zero-based, so rows 43 to 66)
ChbaghL_df = df.iloc[43:66, :]

# Show this subset
display(ChbaghL_df)

# Now calculate column sums for this subset (numeric columns only)
column_sums = ChbaghL_df.select_dtypes(include='number').sum()

# Convert to a clean DataFrame
column_sums_df = pd.DataFrame(column_sums).T
display(column_sums_df)

In [ ]:
df_ratio_43 = ChbaghL_df.div(ChbaghL_df.sum(axis=1), axis=0)


In [ ]:
df_ratio43.head()

In [ ]:


# Suppose df_ratio_34 is one street with 15 photos (rows)
#This line divides your rows into groups of 5
# Compute mean ratios for each group of 5 photos

groups = np.arange(len(df_ratio_43)) // 5

street_part_means_43 = (
    df_ratio_43.groupby(groups)
    .mean(numeric_only=True)
    .reset_index(drop=True)
)



street_part_means_43





In [ ]:
analyzer = DataAnalyzer("/kaggle/working/droped_output.csv")
df = analyzer.df

# Drop the first row (index 0) and reset index
df = df.drop(0).reset_index(drop=True)

# Drop the first column
df = df.iloc[:, 1:]

# Select only rows 2 to 15 (Python index is zero-based, so rows 1 to 14)
ChbaghMid_df = df.iloc[66:81, :]

# Show this subset
display(ChbaghMid_df)

# Now calculate column sums for this subset (numeric columns only)
column_sums = ChbaghMid_df.select_dtypes(include='number').sum()

# Convert to a clean DataFrame
column_sums_df = pd.DataFrame(column_sums).T
display(column_sums_df)

In [ ]:
df_ratio_66 = ChbaghMid_df.div(ChbaghMid_df.sum(axis=1), axis=0)


In [ ]:
# Suppose df_ratio_66 is one street with  photos (rows)
df_ratio_66["group_id"] = np.arange(len(df_ratio_66)) // 5  # groups of 5 rows

#This line divides your rows into groups of 5
# Compute mean ratios for each group of 5 photos

street_part_means_66 = (
    df_ratio_66.groupby("group_id")
    .mean(numeric_only=True)
    .reset_index()
)

street_part_means_66


In [ ]:
analyzer = DataAnalyzer("/kaggle/working/droped_output.csv")
df = analyzer.df

# Drop the first row (index 0) and reset index
df = df.drop(0).reset_index(drop=True)

# Drop the first column
df = df.iloc[:, 1:]

# Select only rows 2 to 15 (Python index is zero-based, so rows 81 to 117)
ChbaghR_df = df.iloc[81:117, :]

# Show this subset
display(ChbaghR_df)

# Now calculate column sums for this subset (numeric columns only)
column_sums = ChbaghR_df.select_dtypes(include='number').sum()

# Convert to a clean DataFrame
column_sums_df = pd.DataFrame(column_sums).T
display(column_sums_df)

In [ ]:
df_ratio_81 = ChbaghR_df.div(ChbaghR_df.sum(axis=1), axis=0)


In [ ]:
display(df_ratio_81)

In [ ]:
street_mean_ratio_81 = df_ratio_81.mean(axis=0)
display(street_mean_ratio_81)

In [ ]:

# Plot a pie chart
street_mean_ratio_81.plot(
    kind='pie',
    autopct='%1.1f%%',       # Show percentages
    figsize=(6, 6),
    startangle=90,           # Rotate so first slice starts at top
    ylabel=''                # Remove y-axis label
)

plt.title("Mean Visual Composition of Street")
plt.show()


In [ ]:
# Suppose df_ratio_81 is one street with  photos (rows)
df_ratio_81["group_id"] = np.arange(len(df_ratio_81)) // 5  # groups of 5 rows
#This line divides your rows into groups of 5
# Compute mean ratios for each group of 5 photos
street_part_means_81 = (
    df_ratio_81.groupby("group_id")
    .mean(numeric_only=True)
    .reset_index())

street_part_means_81


In [ ]:
analyzer = DataAnalyzer("/kaggle/working/droped_output.csv")
df = analyzer.df

# Drop the first row (index 0) and reset index
df = df.drop(0).reset_index(drop=True)

# Drop the first column
df = df.iloc[:, 1:]

# Select only rows 2 to 15 (Python index is zero-based, so rows 1 to 14)
NagJaAx_df = df.iloc[126:132, :]

# Show this subset
display(NagJaAx_df)

# Now calculate column sums for this subset (numeric columns only)
column_sums = NagJaAx_df.select_dtypes(include='number').sum()

# Convert to a clean DataFrame
column_sums_df = pd.DataFrame(column_sums).T
display(column_sums_df)

In [ ]:
df_ratio_126 = NagJaAx_df.div(NagJaAx_df.sum(axis=1), axis=0)


In [ ]:
display(df_ratio_126)

In [ ]:
street_mean_ratio_126 = df_ratio_126.mean(axis=0)
display(street_mean_ratio_126)

In [ ]:
# Suppose df_ratio_81 is one street with  photos (rows)
df_ratio_126["group_id"] = np.arange(len(df_ratio_126)) // 3  # groups of 3 rows
#This line divides your rows into groups of 2
# Compute mean ratios for each group of 5 photos
street_part_means_126 = (
    df_ratio_126.groupby("group_id")
    .mean(numeric_only=True)
    .reset_index()
)

street_part_means_126


In [ ]:
analyzer = DataAnalyzer("/kaggle/working/droped_output.csv")
df = analyzer.df

# Drop the first row (index 0) and reset index
df = df.drop(0).reset_index(drop=True)

# Drop the first column
df = df.iloc[:, 1:]

# Select only rows 2 to 15 (Python index is zero-based, so rows 1 to 14)
NagJaSWalk_df = df.iloc[132:146, :]

# Show this subset
display(NagJaSWalk_df)

# Now calculate column sums for this subset (numeric columns only)
column_sums = NagJaSWalk_df.select_dtypes(include='number').sum()

# Convert to a clean DataFrame
column_sums_df = pd.DataFrame(column_sums).T
display(column_sums_df)

In [ ]:
df_ratio_132 = NagJaSWalk_df.div(NagJaSWalk_df.sum(axis=1), axis=0)


In [ ]:
street_mean_ratio_132 = df_ratio_132.mean(axis=0)
display(street_mean_ratio_132)

In [ ]:
# Suppose df_ratio_81 is one street with  photos (rows)
df_ratio_132["group_id"] = np.arange(len(df_ratio_132)) // 5  # groups of 5 rows
#This line divides your rows into groups of 2
# Compute mean ratios for each group of 5 photos
street_part_means_132 = (
    df_ratio_132.groupby("group_id")
    .mean(numeric_only=True)
    .reset_index()
)

street_part_means_132


In [ ]:
analyzer = DataAnalyzer("/kaggle/working/droped_output.csv")
df = analyzer.df

# Drop the first row (index 0) and reset index
df = df.drop(0).reset_index(drop=True)

# Drop the first column
df = df.iloc[:, 1:]

# Select only rows 2 to 15 (Python index is zero-based, so rows 1 to 14)
OstanSt_df = df.iloc[146:163, :]

# Show this subset
display(OstanSt_df)

# Now calculate column sums for this subset (numeric columns only)
column_sums = OstanSt_df.select_dtypes(include='number').sum()

# Convert to a clean DataFrame
column_sums_df = pd.DataFrame(column_sums).T
display(column_sums_df)

In [ ]:
df_ratio_146 = OstanSt_df.div(OstanSt_df.sum(axis=1), axis=0)


In [ ]:
display(df_ratio_146)

In [ ]:
# Suppose df_ratio_81 is one street with  photos (rows)
df_ratio_146["group_id"] = np.arange(len(df_ratio_146)) // 5  # groups of 5 rows
#This line divides your rows into groups of 2
# Compute mean ratios for each group of 5 photos
street_part_means_146 = (
    df_ratio_146.groupby("group_id")
    .mean(numeric_only=True)
    .reset_index()
)

street_part_means_146

In [ ]:
# Drop the first row (index 0) and reset index
df = df.drop(0).reset_index(drop=True)

# Drop the first column
df = df.iloc[:, 1:]

# Select only rows 2 to 15 (Python index is zero-based, so rows 1 to 14)
SepahL_df = df.iloc[174:200, :]


# Count missing values in each column
#you can use .isnull().sum() to check how many missing values are in each column, then find which one has the most.
missing_counts = df.isnull().sum()

# Print all missing counts
print("Missing values per column:\n", missing_counts)

# Find the column with the most missing values
max_missing_col = missing_counts.idxmax()
max_missing_val = missing_counts.max()

print(f"\nColumn with the most missing values: {max_missing_col} → {max_missing_val} missing")

In [ ]:
from sklearn.impute import KNNImputer
# sklearn.impute is a module (sub-library) inside scikit-learn (sklearn) that provides tools for handling missing data (imputation).

# Select numeric columns for imputation
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns

# Create a copy of numeric data
df_numeric = df[numeric_cols]

# Initialize KNN imputer (you can change n_neighbors=3 → 5 or 7)
imputer = KNNImputer(n_neighbors=3)

# Fit and transform
df_imputed = imputer.fit_transform(df_numeric)

# Convert back to DataFrame
df_imputed = pd.DataFrame(df_imputed, columns=numeric_cols)

# Replace original numeric columns with imputed ones
df[numeric_cols] = df_imputed

print("Missing values imputed using KNN.")
print(df.isnull().sum())


In [ ]:
imputer = KNNImputer(n_neighbors=10)


In [ ]:
analyzer = DataAnalyzer("/kaggle/working/droped_output.csv")
df = analyzer.df

# Drop the first row (index 0) and reset index
df = df.drop(0).reset_index(drop=True)

# Drop the first column
df = df.iloc[:, 1:]

# Select only rows 2 to 15 (Python index is zero-based, so rows 1 to 14)
SepahL_df = df.iloc[163:174, :]

# Show this subset
display(SepahL_df)

# Now calculate column sums for this subset (numeric columns only)
column_sums = SepahL_df.select_dtypes(include='number').sum()

# Convert to a clean DataFrame
column_sums_df = pd.DataFrame(column_sums).T
display(column_sums_df)

In [ ]:
df_ratio_162 = SepahL_df.div(SepahL_df.sum(axis=1), axis=0)


In [ ]:
display(df_ratio_162)

In [ ]:


# Suppose df_ratio_81 is one street with  photos (rows)
df_ratio_162["group_id"] = np.arange(len(df_ratio_162)) // 5  # groups of 5 rows
#This line divides your rows into groups of 2
# Compute mean ratios for each group of 5 photos
street_part_means_162 = (
    df_ratio_162.groupby("group_id")
    .mean(numeric_only=True)
    .reset_index()
)

street_part_means_162


In [ ]:
analyzer = DataAnalyzer("/kaggle/working/droped_output.csv")
df = analyzer.df

# Drop the first row (index 0) and reset index
df = df.drop(0).reset_index(drop=True)

# Drop the first column
df = df.iloc[: , 1:]

# Select only rows 2 to 15 (Python index is zero-based, so rows 1 to 14)
SepahMid_df = df.iloc[174:187 , :]

# Show this subset
display(SepahMid_df)

# Now calculate column sums for this subset (numeric columns only)
column_sums = SepahMid_df.select_dtypes(include='number').sum()

# Convert to a clean DataFrame
column_sums_df = pd.DataFrame(column_sums).T
display(column_sums_df)

In [ ]:
df_ratio_174 = SepahMid_df.div(SepahMid_df.sum(axis=1), axis=0)
display(df_ratio_174)

In [ ]:


# Suppose df_ratio_81 is one street with  photos (rows)
df_ratio_174["group_id"] = np.arange(len(df_ratio_174)) // 5  # groups of 5 rows
#This line divides your rows into groups of 2
# Compute mean ratios for each group of 5 photos
street_part_means_174 = (
    df_ratio_174.groupby("group_id")
    .mean(numeric_only=True)
    .reset_index()
)

street_part_means_174

In [ ]:
analyzer = DataAnalyzer("/kaggle/working/droped_output.csv")
df = analyzer.df

# Drop the first row (index 0) and reset index
df = df.drop(0).reset_index(drop=True)

# Drop the first column
df = df.iloc[: , 1:]

# Select only rows 2 to 15 (Python index is zero-based, so rows 1 to 14)
SepahR_df = df.iloc[187:194 , :]

# Show this subset
display(SepahR_df)

# Now calculate column sums for this subset (numeric columns only)
column_sums = SepahR_df.select_dtypes(include='number').sum()

# Convert to a clean DataFrame
column_sums_df = pd.DataFrame(column_sums).T
display(column_sums_df)

# Making numeric new columns for correlation matrix 

In [ ]:
df_ratio_187 = SepahR_df.div(SepahR_df.sum(axis=1), axis=0)
display(df_ratio_187)

In [ ]:


# Suppose df_ratio_81 is one street with  photos (rows)
df_ratio_187["group_id"] = np.arange(len(df_ratio_187)) // 5  # groups of 5 rows
#This line divides your rows into groups of 2
# Compute mean ratios for each group of 5 photos
street_part_means_187 = (
    df_ratio_187.groupby("group_id")
    .mean(numeric_only=True)
    .reset_index()
)

street_part_means_187

In [ ]:
# Combine all grouped mean DataFrames
all_streets_groups = pd.concat([
    street_part_means_15,
    street_part_means_34,
    street_part_means43,
    street_part_means_43,
    street_part_means_66,
    street_part_means_81,
    street_part_means_126,
    street_part_means_132,
    street_part_means_146,
    street_part_means_162,
    street_part_means_174,
    street_part_means_187,
    
], ignore_index=True)


In [ ]:
numeric_df = all_streets_groups.select_dtypes(include='number')


In [ ]:
print("Numeric columns:", numeric_df.columns.tolist())
print("Shape of numeric_df:", numeric_df.shape)


# Target class, Imageability

In [ ]:
 # we make the Imageability column as formula below and with coeeficient 

numeric_df["Imageability"] = (
   numeric_df["Trees with columnar shape"] +
   numeric_df["Water"] +
   numeric_df["Wall_pixels"] +
   numeric_df["merge_E_H_N"]
)


In [ ]:

# imageability for 12 streets and each street about  3 to 4 means. 
display(numeric_df["Imageability"])

In [ ]:
imageability_corr = numeric_df.corr(method='pearson')["Imageability"].sort_values(ascending=False)
display(imageability_corr)


In [ ]:
numeric_df = all_streets_groups.select_dtypes(include='number')

# Drop unwanted numeric columns
numeric_df = numeric_df.drop(columns=["Flower", "Flower box","Trees with oval shape","Class_34","group_id","Dome"], errors="ignore")

correlation_matrix = numeric_df.corr(method='pearson')
display(correlation_matrix)


In [ ]:
 # create a separate dataframe for correlation to not reappear the dropped column in cooreelation df 
correlation_matrix = numeric_df.corr(method='pearson')
display(correlation_matrix)


In [ ]:
# corr_df is new name for df after dropping the unwanted column whoch was numeric_df
# we Start with a fresh copy to drop all unwnted and again make correlation again against imageability
corr_df = all_streets_groups.select_dtypes(include='number').copy()


In [ ]:
#Drop columns from corr_df only
# List of columns you want to drop
cols_to_drop = [
    "Flower",
    "Flower box",
    "Trees with oval shape",
    "Class_34",
    "group_id",
    "Dome"
]

# Make separate dataframe
corr_df = numeric_df.copy()

# Drop columns safely
corr_df.drop(columns=cols_to_drop, errors="ignore", inplace=True)


In [ ]:
#Add Imageability to corr_df
corr_df["Imageability"] = (
     corr_df["Trees with columnar shape"] +
    corr_df["Water"] +
    0.25 * corr_df["Wall_pixels"] +
     corr_df["merge_E_H_N"]
)


In [ ]:

#Move Imageability to last column
corr_df = corr_df[[c for c in corr_df.columns if c != "Imageability"] + ["Imageability"]]



In [ ]:
corr_df.columns.tolist()


In [ ]:
#compute Pearson correlation ONLY from corr_df
correlation_matrix = corr_df.corr(method="pearson")
display(correlation_matrix)


In [ ]:
from scipy.stats import pearsonr

corrs = {}
pvals = {}

for col1 in numeric_df.columns:
    for col2 in numeric_df.columns:
        r, p = pearsonr(numeric_df[col1], numeric_df[col2])
        corrs[(col1, col2)] = r
        pvals[(col1, col2)] = p

corr_df = pd.DataFrame(corrs, index=["r"]).T.unstack().unstack()
pval_df = pd.DataFrame(pvals, index=["p"]).T.unstack().unstack()


In [ ]:
display(corr_df)
display(pval_df)


# Replace NaN values before comparison:otherwise py:73: RuntimeWarning: invalid value encountered

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)


In [ ]:
pval_df = pval_df.fillna(1.0)  # assume non-significant where p-value is NaN
significant_corrs = corr_df.where(pval_df < 0.05)


# Heatmap correlation from seaborn

In [ ]:
import seaborn as sns

plt.figure(figsize=(10,8))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Pearson Correlation Between Visual Features (All Streets)")
plt.show()


### This code hides the NaN cells instead of coloring them, resulting in a cleaner plot.

In [ ]:
# Ensure indices and columns match
pval_df = pval_df.reindex_like(corr_df)

# Then filter correlations with significant p-values (< 0.05)
significant_corrs = corr_df.where(pval_df < 0.05)
significant_corrs = corr_df[(pval_df < 0.05)]


In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(correlation_matrix, 
            annot=True, fmt=".2f", cmap="coolwarm", center=0, 
            mask=correlation_matrix.isnull())
plt.title("Pearson Correlation Between Visual Features (All Streets)")
plt.show()


In [ ]:
# Save the correlation matrix to a CSV file
correlation_matrix.to_csv("/kaggle/working/correlation_matrix.csv", index=True)

print("Correlation matrix saved to: /kaggle/working/correlation_matrix.csv")